In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from io import StringIO
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

In [ ]:
years = [2023, 2024, 2025] ## get list of years to import 
base_url = "https://www.warrennolan.com/softball/{}/sos-rpi" ## get url

In [ ]:
all_years_data = []

for year in years:
    url = base_url.format(year)
    response = requests.get(url)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Find the first table on the page (it's already nicely formatted)
    table = soup.find('table')

    # Convert the table to a dataframe
    df = pd.read_html(str(table))[0]
    df["Year"] = year  # Add year column

    all_years_data.append(df)

# Combine all years into one dataframe
sos = pd.concat(all_years_data, ignore_index=True)

# Preview the combined dataframe
print(sos.head())


In [ ]:
sos.to_csv("sos.csv")

In [ ]:
years_nonconf = list(range(2021, 2026)) ## get list of years to import 
base_url = "https://d1softball.com/nitty-gritty/?season={}" ## get url

In [ ]:
# Set up headless browser
options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)

In [ ]:
all_years_data = []

for year in years_nonconf:
    driver.get(base_url.format(year))
    time.sleep(2)

    frames = []
    while True:
        # Read whatever table is currently visible
        tables = pd.read_html(driver.page_source)
        if not tables:
            break
        for table in tables:
            frames.append(table.assign(Year=year))


        # Attempt to click "Next" or "Show More"
        try:
            nxt = driver.find_element(By.XPATH, "//a[text()='Next' or text()='›' or text()='>']")
            # If it's disabled, stop
            if "disabled" in nxt.get_attribute("class").lower():
                break
            nxt.click()
            time.sleep(2)
        except:
            break

    if frames:
        all_years_data.append(pd.concat(frames, ignore_index=True))

driver.quit()

# Combine into one DataFrame and export to HTML
nonconf = pd.concat(all_years_data, ignore_index=True)
#nonconf.to_html("nonconf_all_years.html", index=False)

print(nonconf.head())

In [ ]:
nonconf.to_csv("nonconf_wp.csv")